# *This notebook provides a workflow to determine the best hyperparameters to build a Random Forest (RF) model before running a RF classification using them. Three hyperparameters are optimized*:
- *N_estimators*
- *Max_features*
- *Max_depth*

The results from the final model can then be compared with the results of the Personalized Pagerank (PPR) and Support Vector Machines (SVM) models in the context of a benchmark for a publication.

### *Importing the required libraries*

In [1]:
import sys
sys.path.append("../scripts")

import argparse
import glob
import networkx as nx
import numpy as np
import os
import pandas as pd
import random_forest_benchmark
import shutil
import useful_functions
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from tqdm import tqdm

### *Reading the training genes and setting the input parameters*

In [ ]:
# Reading the training genes
genes = pd.read_csv("../TrainingGenes/training_genes_stalk_cell.csv")
genes = genes["Feature"].to_list()

# Setting the input dataset
dataset = "STRING_CS_100_E-GEOD-45750_corr_07"

# Setting the process of interest
process = "stalk_cell"

# Setting the output folder
dir_RF = f"../results/{process}/{dataset}"

if not os.path.exists(dir_RF):
    os.mkdir(dir_RF)

dir_RF = f"../results/{process}/{dataset}/RF"

if not os.path.exists(dir_RF):
    os.mkdir(dir_RF)

dir_benchmark = f"{dir_RF}/Benchmark_hyperparameters"
if not os.path.exists(dir_benchmark):
    os.mkdir(dir_benchmark)
else:
    shutil.rmtree(dir_benchmark)
    os.mkdir(dir_benchmark)

# Reading the input graph
graph = nx.read_graphml(f"../graphs/integrated/integrated/{dataset}.graphml")

# Turning the input graph into a matrix which will be used as input data for RF
data_graph = useful_functions.graph_to_matrix(graph)
print(f"Graph converted to a matrix with shape: {data_graph.shape}")
print(data_graph.head())

# Checking if the graph contains the training genes
valid_genes = [gene for gene in genes if gene in graph.nodes]
print(f"Number of training genes in the graph: {len(valid_genes)}")

### *Determining the best hyperparameters*

In [ ]:
# Running a RF algorithm on a 5 fold CV
random_forest_benchmark.benchmark_hyperparameters_rf(data_graph, valid_genes, dir_benchmark)

# Creating empty lists to store the best parameters results
N_estimators = []
Max_features = []
Max_depth = []
AUCs = []

# Retrieving the best parameters in every file
files = glob.glob(f"{dir_benchmark}/*/random_forest_performance_summary.csv")
for file in files:
    df = pd.read_csv(file)
    
    parameters = file.split("\\")[2]  # This line of code needs to be adapted depending on the path
    n_estimators = parameters.split("_")[2]
    max_features = parameters.split("_")[5]
    max_depth = parameters.split("_")[8]
    auc = float(df["roc_auc"].values[5])

    N_estimators.append(n_estimators)
    Max_features.append(max_features)
    Max_depth.append(max_depth)
    AUCs.append(auc)

# Compiling all the results in a dataframe
df_benchmark = pd.DataFrame({"N_estimators": N_estimators,
                            "Max_features": Max_features,
                            "Max_depth": Max_depth,
                            "AUC": AUCs})

df_benchmark.to_csv(f"{dir_benchmark}/RF_benchmark_results.csv",
                   sep = ",", index = False)

# Finding the best hyperparameters
best_hyperparameters = df_benchmark[df_benchmark["AUC"] == df_benchmark["AUC"].max()]
best_hyperparameters.to_csv(f"{dir_benchmark}/RF_best_hyperparameters.csv",
                           sep = ",", index = False)

### *Running a RF classification with the best hyperparameters*

In [ ]:
# Creating the folder for the 5 fold-CV
dir_5_fold = f"../results/{process}/{dataset}/RF/5_fold-CV"
if not os.path.exists(dir_5_fold):
    os.mkdir(dir_5_fold)
else:
    shutil.rmtree(dir_5_fold)
    os.mkdir(dir_5_fold)

# Reading the best hyperparameters file and storing them invariables
df_hyperparameters = pd.read_csv(f"../results/{process}/{dataset}/RF/Benchmark_hyperparameters/RF_best_hyperparameters.csv")
best_n_estimators = int(df_hyperparameters["N_estimators"][0])
best_features = df_hyperparameters["Max_features"][0]
best_depth = int(df_hyperparameters["Max_depth"][0])

# Turning it into a matrix which will be used as input data for RF
data_graph = useful_functions.graph_to_matrix(graph)
print(f"Graph converted to a matrix with shape: {data_graph.shape}")
print(data_graph.head())

# Checking if the graph contains the training genes
valid_genes = [gene for gene in genes if gene in graph.nodes]
print(f"Number of training genes in the graph: {len(valid_genes)}")

# Running a RF algorithm on a 5 fold CV
print("Running Random Forest analysis ...")
random_forest_benchmark.random_forest_5Fold_CV(data_graph, 
                                               valid_genes, 
                                               dir_5_fold, 
                                               best_n_estimators,
                                               best_features,
                                               best_depth)